In [24]:
from importlib.metadata import version
print(f"sagemaker: {version('sagemaker')}")
print(f"boto3: {version('boto3')}")
print(f"botocore: {version('botocore')}")

sagemaker: 3.5.0
boto3: 1.42.58
botocore: 1.42.58


In [43]:
# initialize constants
REGION = "us-west-2"
TRAINING_JOB_NAME = "test-lora-training-1-1773273846617"
INSTANCE_TYPE = "ml.g5.8xlarge"
ROLE="arn:aws:iam::099324990371:role/service-role/AmazonSageMaker-ExecutionRole-20260219T233135"

LMI_IMAGE_URI = "763104351884.dkr.ecr.us-west-2.amazonaws.com/djl-inference:0.34.0-lmi16.0.0-cu128"
LMI_IMAGE_URI_31 = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/djl-inference:0.31.0-lmi13.0.0-cu124"

BASE_MODEL_S3_URI = f"s3://jumpstart-private-cache-prod-{REGION}/meta-textgeneration/meta-textgeneration-llama-3-2-1b-instruct/artifacts/inference-prepack/v1.0.0/"

import random
name_suffix = random.randint(100, 10000)

In [44]:
# initialize clients
import boto3
sm = boto3.client("sagemaker", region_name=REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=REGION)

In [45]:
# get s3 artifact location
response = sm.describe_training_job(TrainingJobName=TRAINING_JOB_NAME)
model_s3_uri = response["ModelArtifacts"]["S3ModelArtifacts"]
adapter_s3_uri = f"{model_s3_uri}/checkpoints/hf/"
print(f"  Adapter weights: {adapter_s3_uri}")

  Adapter weights: s3://sagemaker-us-west-2-099324990371/model-customization/output-artifacts/test-lora-training-1-1773273846617/output/model/checkpoints/hf/


In [ ]:
# create model in SageMaker using boto3
model_name = f"model-{name_suffix}"
model = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=ROLE,
    PrimaryContainer={
        "Image": LMI_IMAGE_URI,
        "Environment": {
            # Use lmi-dist for rolling batch — NOT "disable" (which requires the vllm entrypoint on 0.34.0+)
            "OPTION_ROLLING_BATCH": "lmi-dist",
            "OPTION_ENABLE_LORA": "true",
            "OPTION_MAX_LORAS": "8",
            "OPTION_MAX_CPU_LORAS": "64",
            "OPTION_MAX_LORA_RANK": "128",
            "OPTION_MAX_ROLLING_BATCH_SIZE": "8",
            # Must match GPU count on the instance: "1" for g5.2xlarge, "8" for g6e.48xlarge
            "OPTION_TENSOR_PARALLEL_DEGREE": "1",
            "OPTION_DTYPE": "fp16",
            "OPTION_MAX_MODEL_LEN": "4096",
        },
        # Load base model from JumpStart S3 cache — avoids needing HF_TOKEN for gated models
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": BASE_MODEL_S3_URI,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
                "ModelAccessConfig": {"AcceptEula": True},
            }
        },
    },
)
print(f"model: {model}")

model: {'ModelArn': 'arn:aws:sagemaker:us-west-2:099324990371:model/model-1977', 'ResponseMetadata': {'RequestId': '510a2671-03c1-464a-81d1-1ad7f512a72d', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '510a2671-03c1-464a-81d1-1ad7f512a72d', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '72', 'date': 'Thu, 12 Mar 2026 20:09:45 GMT'}, 'RetryAttempts': 0}}


In [ ]:
# create model in SageMaker using ModelBuilder
from sagemaker.core.resources import TrainingJob
from sagemaker.serve import ModelBuilder

training_job = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
print(f"model package arn: {training_job.output_model_package_arn}")

model_name = f"model-{name_suffix}"
model_builder = ModelBuilder(model=training_job, role_arn=ROLE, instance_type=INSTANCE_TYPE)
model = model_builder.build(model_name=model_name)
print(f"model arn: {model.model_arn}")

In [30]:
# create endpoint
endpoint_name = f"e2e-{name_suffix}"
ep_config = sm.create_endpoint_config(
    EndpointConfigName=endpoint_name,
    ExecutionRoleArn=ROLE,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "InstanceType": INSTANCE_TYPE,
            "InitialInstanceCount": 1,
        }
    ],
)
print(f"endpoint config: {ep_config}")

endpoint = sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_name)
print(f"endpoint: {endpoint}")

endpoint config: {'EndpointConfigArn': 'arn:aws:sagemaker:us-west-2:099324990371:endpoint-config/e2e-1977', 'ResponseMetadata': {'RequestId': 'ded5b4e4-1b3e-49c2-97c2-25bb7a11b2a4', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'ded5b4e4-1b3e-49c2-97c2-25bb7a11b2a4', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '89', 'date': 'Thu, 12 Mar 2026 20:10:22 GMT'}, 'RetryAttempts': 0}}
endpoint: {'EndpointArn': 'arn:aws:sagemaker:us-west-2:099324990371:endpoint/e2e-1977', 'ResponseMetadata': {'RequestId': 'ca82b575-df5e-48f7-ba9b-74de1137d1ef', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'ca82b575-df5e-48f7-ba9b-74de1137d1ef', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options

In [31]:
# create base model inference component
base_ic_name = f"{endpoint_name}-inference-component"
base_inference_component = sm.create_inference_component(
    InferenceComponentName=base_ic_name,
    EndpointName=endpoint_name,
    VariantName="AllTraffic",
    Specification={
        "ModelName": model_name,
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": 4096,
            "NumberOfAcceleratorDevicesRequired": 1,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)
print(f"base inference component: {base_inference_component}")

base inference component: {'InferenceComponentArn': 'arn:aws:sagemaker:us-west-2:099324990371:inference-component/e2e-1977-inference-component', 'ResponseMetadata': {'RequestId': 'c24a105b-762b-4aa1-b17c-ae71adfd0df2', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'c24a105b-762b-4aa1-b17c-ae71adfd0df2', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '117', 'date': 'Thu, 12 Mar 2026 20:10:28 GMT'}, 'RetryAttempts': 0}}


In [46]:
# create adapter inference component
endpoint_name = "e2e-1977"
base_ic_name = "e2e-1977-inference-component"

adapter_ic_name = f"{endpoint_name}-adapter"
adapter_inference_component = sm.create_inference_component(
    InferenceComponentName=adapter_ic_name,
    EndpointName=endpoint_name,
    Specification={
        "BaseInferenceComponentName": base_ic_name,
        "Container": {"ArtifactUrl": adapter_s3_uri},
    },
)
print(f"adapter inference component: {adapter_inference_component}")

adapter inference component: {'InferenceComponentArn': 'arn:aws:sagemaker:us-west-2:099324990371:inference-component/e2e-1977-adapter', 'ResponseMetadata': {'RequestId': 'fd3e04bf-1b48-4211-bd92-16d3d67cc4df', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'fd3e04bf-1b48-4211-bd92-16d3d67cc4df', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '105', 'date': 'Thu, 12 Mar 2026 20:39:33 GMT'}, 'RetryAttempts': 0}}


In [38]:
# test inference on base model inference component
import json
payload = json.dumps({"inputs": "What is the capital of France?", "parameters": {"max_new_tokens": 50}})
base_model_response = sm_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    InferenceComponentName=base_ic_name,
    Body=payload,
    ContentType="application/json",
)
result = json.loads(base_model_response["Body"].read().decode())
print(f"Response: {result}")

Response: {'generated_text': ' Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris'}


In [54]:
from pprint import pprint

# resp = sm.describe_inference_component(InferenceComponentName=adapter_ic_name)
resp = sm.describe_inference_component(InferenceComponentName="e2e-5429-inference-component")
# resp = sm.describe_endpoint(EndpointName="e2e-5429")

# status = resp["InferenceComponentStatus"]
# print(f"Status: {status}")
pprint(resp)

{'CreationTime': datetime.datetime(2026, 3, 12, 9, 50, 14, 970000, tzinfo=tzlocal()),
 'EndpointArn': 'arn:aws:sagemaker:us-west-2:099324990371:endpoint/e2e-5429',
 'EndpointName': 'e2e-5429',
 'InferenceComponentArn': 'arn:aws:sagemaker:us-west-2:099324990371:inference-component/e2e-5429-inference-component',
 'InferenceComponentName': 'e2e-5429-inference-component',
 'InferenceComponentStatus': 'InService',
 'LastModifiedTime': datetime.datetime(2026, 3, 12, 9, 55, 36, 65000, tzinfo=tzlocal()),
 'ResponseMetadata': {'HTTPHeaders': {'cache-control': 'no-cache, no-store, '
                                                       'must-revalidate',
                                      'content-length': '973',
                                      'content-security-policy': 'frame-ancestors '
                                                                 "'none'",
                                      'content-type': 'application/x-amz-json-1.1',
                                      'd

In [ ]:
# test inference on adapter inference component
import json
payload = json.dumps({"inputs": "What is the capital of France?", "parameters": {"max_new_tokens": 50}})
adapter_response = sm_runtime.invoke_endpoint(
    EndpointName="e2e-5429", #endpoint_name,
    InferenceComponentName="e2e-5429-inference-component", #adapter_ic_name,
    Body=payload,
    ContentType="application/json",
)
result = json.loads(adapter_response["Body"].read().decode())
print(f"  Response: {result}")

  Response: {'generated_text': ' Paris.\nWhat is the capital of the United States? Washington, D.C.\nWhat is the capital of the United Kingdom? London.\nWhat is the capital of Australia? Canberra.\nWhat is the capital of Canada? Ottawa.\nWhat is the capital of'}


In [37]:
from pprint import pprint

tags = sm.list_tags(ResourceArn=f"arn:aws:sagemaker:{REGION}:099324990371:training-job/{TRAINING_JOB_NAME}")
jumpstart_model_id = next(t["Value"] for t in tags["Tags"] if t["Key"] == "sagemaker-studio:jumpstart-model-id")
print(jumpstart_model_id)

hub_resp = sm.describe_hub_content(
    HubName="SageMakerPublicHub", HubContentType="Model", HubContentName=jumpstart_model_id
)
hub_doc = json.loads(hub_resp["HubContentDocument"])
pprint(hub_doc)
config = hub_doc["InferenceConfigComponents"]["lmi"]
base_model_s3_uri = config["HostingArtifactUri"]
instance_family = INSTANCE_TYPE.split(".")[1]
lmi_image_uri = config["HostingInstanceTypeVariants"]["Variants"][instance_family]["Properties"]["ImageUri"]

meta-textgeneration-llama-3-2-1b-instruct
{'Capabilities': ['TRAINING', 'FINE_TUNING', 'VALIDATION', 'CUSTOMIZATION'],
 'ContainerStartupHealthCheckTimeout': 1200,
 'ContextualHelp': {'HubDefaultTrainData': ["Dataset: [OpenAssistant's TOP-1 "
                                            'Conversation '
                                            'Threads](https://huggingface.co/datasets/OpenAssistant/oasst_top1_2023-08-25)',
                                            "OpenAssistant's TOP-1 "
                                            'Conversation Threads dataset '
                                            'contains roughly 13,000 samples '
                                            'of conversations between the '
                                            'Assistant and the user.',
                                            'License: [Apache '
                                            '2.0](https://jumpstart-cache-prod-us-east-2.s3-us-east-2.amazonaws.com/licenses/Apache-Licen

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:12                                                                                   │
│                                                                                                  │
│    9 )                                                                                           │
│   10 hub_doc = json.loads(hub_resp["HubContentDocument"])                                        │
│   11 pprint(hub_doc)                                                                             │
│ ❱ 12 config = hub_doc["InferenceConfigComponents"]["lmi"]                                        │
│   13 base_model_s3_uri = config["HostingArtifactUri"]                                            │
│   14 instance_family = INSTANCE_TYPE.split(".")[1]                                               │
│   15 lmi_image_uri = config["HostingInstanceTypeVariants"]["Variants"][instance_family]["Prop    │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyError: 'InferenceConfigComponents'